In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# Function to read CSV files
def read_csv_file(filename, usecols=None, encoding='ISO-8859-1'):
    return pd.read_csv(filename, usecols=usecols, encoding=encoding)

# Load initial datasets
df_v2i = read_csv_file('V2I.csv')
df_v1e = read_csv_file('V1E.CSV')
df_v3j = read_csv_file('V3J.csv')
df_v1a = read_csv_file('V1A.CSV', usecols=['V1AH04', 'V1AH05', 'V1AH07', 'V1AH08', 'PublicID'])
df_v3a = read_csv_file('V3A.CSV', usecols=['V3AG04', 'V3AG05', 'V3AG07', 'V3AG08', 'PublicID'])
df_v2a = read_csv_file('V2A.csv', usecols=['V2AG03a', 'V2AG03b', 'V2AG03c', 'V2AG03d', 'V2AG03e', 'V2AG03f', 'V2AG03g', 'V2AG03h', 'V2AG03i', 'PublicID'])
df_v1c = read_csv_file('V1C.CSV')
df_v1h = read_csv_file('V1H.CSV')
df_cma = read_csv_file('CMA.CSV')
df_v1g = read_csv_file('V1G.csv', usecols=['V1GA01', 'V1GA02', 'V1GA03', 'V1GA04', 'V1GA05', 'V1GA06', 'V1GA07', 'V1GA08', 'V1GA09', 'V1GA10', 'V1GA11', 'V1GA12', 'PublicID'])
df_outcome = read_csv_file('outcome.csv', usecols=['PublicID', 'MH_outcome'])


C:\Users\91799\AppData\Local\Temp\ipykernel_2588\2769153903.py:15: DtypeWarning: Columns (4,27,28,30,52,56,58,60,62,64,66,68,85,86,88,99,100,101,102,103,104,105,113,115,116,142,143,144,145,146,147,148,149,151,160,172,178,222,224,246,252,288,292,293,294,296,314,325,326,327,333,343,367,381,387,388,391,393,402,404,405,407,445,448,449,451,454,455,456,485,486,501,502,510,511,519,520) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(filename, usecols=usecols, encoding=encoding)


In [4]:
# Feature engineering for V2I
def compute_resilience_score(df):
    def resilience_level(score):
        if score <= 75:
            return 3
        elif score <= 100:
            return 2
        else:
            return 1

    df['TotalScore'] = df[['V2IA{:02d}'.format(i) for i in range(1, 26)]].sum(axis=1)
    df['ResilienceLevel'] = df['TotalScore'].apply(resilience_level)
    return df[['PublicID', 'TotalScore', 'ResilienceLevel']]

df_v2i_features = compute_resilience_score(df_v2i)

# Feature engineering for V1E
df_v1e['stress_average'] = df_v1e[['V1EA01', 'V1EA02a', 'V1EA02b', 'V1EA02c', 'V1EA02d', 'V1EA02e', 'V1EA02f',
                                    'V1EA02g', 'V1EA02h', 'V1EA02i', 'V1EA02j', 'V1EA02k', 'V1EA02l']].mean(axis=1)
df_v1e_features = df_v1e[['PublicID', 'stress_average']]

# Feature engineering for V3J
def compute_hassles_uplifts(df):
    df['FrequencyOfHassles'] = df[['V3JA02a', 'V3JA02b', 'V3JA02c', 'V3JA02d', 'V3JA02e', 'V3JA02f', 'V3JA02g', 'V3JA02h', 'V3JA02i', 'V3JA02j']].gt(0).sum(axis=1)
    df['FrequencyOfUplifts'] = df[['V3JA01a', 'V3JA01b', 'V3JA01c', 'V3JA01d', 'V3JA01e', 'V3JA01f', 'V3JA01g', 'V3JA01h', 'V3JA01i', 'V3JA01j']].gt(0).sum(axis=1)
    df['IntensityOfHassles'] = df[['V3JA02a', 'V3JA02b', 'V3JA02c', 'V3JA02d', 'V3JA02e', 'V3JA02f', 'V3JA02g', 'V3JA02h', 'V3JA02i', 'V3JA02j']].sum(axis=1) / df['FrequencyOfHassles']
    df['IntensityOfUplifts'] = df[['V3JA01a', 'V3JA01b', 'V3JA01c', 'V3JA01d', 'V3JA01e', 'V3JA01f', 'V3JA01g', 'V3JA01h', 'V3JA01i', 'V3JA01j']].sum(axis=1) / df['FrequencyOfUplifts']
    df['HassleUpliftFrequencyRatio'] = df['FrequencyOfHassles'] / df['FrequencyOfUplifts']
    df['HassleUpliftIntensityRatio'] = df['IntensityOfHassles'] / df['IntensityOfUplifts']
    return df[['PublicID', 'FrequencyOfHassles', 'FrequencyOfUplifts', 'IntensityOfHassles', 'IntensityOfUplifts', 'HassleUpliftFrequencyRatio', 'HassleUpliftIntensityRatio']]

df_v3j_features = compute_hassles_uplifts(df_v3j)

# Combine all features into a single dataframe for modeling
scoring_df = pd.merge(df_v2i_features, df_v1e_features, on='PublicID', how='inner')
scoring_df = pd.merge(scoring_df, df_v3j_features[['PublicID', 'HassleUpliftIntensityRatio']], on='PublicID', how='inner')
scoring_df = pd.merge(scoring_df, df_outcome, on='PublicID', how='inner')


In [5]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Function to preprocess data
def preprocess_data(df, target_column, drop_columns=None):
    # Drop unnecessary columns
    if drop_columns:
        df = df.drop(drop_columns, axis=1)

    # Split features and target
    X = df.drop(target_column, axis=1)
    y = df[target_column]

    # Impute missing values
    imputer = SimpleImputer(strategy='mean')
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

    # Standardize features
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=X.columns)

    return X_scaled, y

# Function to evaluate models
def evaluate_model(y_true, y_pred, model_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    conf_matrix = confusion_matrix(y_true, y_pred)
    print(f"{model_name} Evaluation Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("Confusion Matrix:")
    print(conf_matrix)
    print("\n")

# Preprocess the data
X, y = preprocess_data(scoring_df, target_column='MH_outcome', drop_columns=['PublicID'])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Handle class imbalance using SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Initialize models
models = {
    'Logistic Regression': LogisticRegression(solver='liblinear'),
    'Random Forest': RandomForestClassifier(),
    'XGBoost': XGBClassifier()
}

# Train and evaluate models
for model_name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test)
    evaluate_model(y_test, y_pred, model_name)

Logistic Regression Evaluation Metrics:
Accuracy: 0.7160
Precision: 0.7204
Recall: 0.7160
F1-score: 0.7174
Confusion Matrix:
[[618 226]
 [179 403]]


Random Forest Evaluation Metrics:
Accuracy: 0.6774
Precision: 0.6754
Recall: 0.6774
F1-score: 0.6762
Confusion Matrix:
[[627 217]
 [243 339]]


XGBoost Evaluation Metrics:
Accuracy: 0.6957
Precision: 0.6930
Recall: 0.6957
F1-score: 0.6938
Confusion Matrix:
[[647 197]
 [237 345]]




In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for Random Forest
rf_param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],   # Number of trees in the forest
    'max_depth': [None, 10, 20, 30],             # Maximum depth of the tree
    'min_samples_split': [2, 5, 10],             # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4],               # Minimum number of samples required to be at a leaf node
    'max_features': ['auto', 'sqrt', 'log2']     # Number of features to consider when looking for the best split
}

# Initialize Random Forest model
rf_model = RandomForestClassifier(random_state=42)

# Setup GridSearchCV
rf_grid_search = GridSearchCV(estimator=rf_model, param_grid=rf_param_grid, 
                              cv=5, n_jobs=-1, verbose=2, scoring='f1_weighted')

# Fit GridSearchCV
rf_grid_search.fit(X_train_resampled, y_train_resampled)

# Best parameters from grid search
best_rf_params = rf_grid_search.best_params_
print("Best Random Forest Hyperparameters:", best_rf_params)

# Evaluate the best Random Forest model
best_rf_model = rf_grid_search.best_estimator_
y_pred_rf_best = best_rf_model.predict(X_test)
evaluate_model(y_test, y_pred_rf_best, "Tuned Random Forest")


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# Define the parameter grid for Logistic Regression
logreg_param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],               # Regularization strength; must be a positive float
    'penalty': ['l2', 'l1', 'elasticnet'],      # Penalty type; 'l1' lasso, 'l2' ridge, 'elasticnet' combination of both
    'solver': ['liblinear', 'saga'],            # Solver; 'liblinear' for small datasets, 'saga' for large datasets
    'l1_ratio': [0, 0.5, 1]                     # L1 ratio; Only used if penalty is 'elasticnet', ignored otherwise
}

# Initialize Logistic Regression model
logistic_model = LogisticRegression(random_state=42, max_iter=1000)

# Setup GridSearchCV
logreg_grid_search = GridSearchCV(estimator=logistic_model, param_grid=logreg_param_grid, 
                                  cv=5, n_jobs=-1, verbose=2, scoring='f1_weighted')

# Fit GridSearchCV
logreg_grid_search.fit(X_train_resampled, y_train_resampled)

# Best parameters from grid search
best_logreg_params = logreg_grid_search.best_params_
print("Best Logistic Regression Hyperparameters:", best_logreg_params)

# Evaluate the best Logistic Regression model
best_logreg_model = logreg_grid_search.best_estimator_
y_pred_logreg_best = best_logreg_model.predict(X_test)
evaluate_model(y_test, y_pred_logreg_best, "Tuned Logistic Regression")
